# Drive times via OSRM (address × pharmacy)

We call a local **OSRM** `table` service to get many-to-many **durations** (seconds) and **distances** (meters) in one HTTP request per batch.

This server caps **100 coordinates** per table request (`Too many table coordinates` beyond that). Coordinates are ordered as **all addresses in the batch, then all pharmacies in the batch**; `sources` / `destinations` split that list so the returned matrix is **rows = addresses**, **columns = pharmacies** for that slice.

**Inputs:** `addresses_with_population_weights.csv` (row order is the matrix row index) and `pharmacies_active.json` (column order follows JSON array order).

**Outputs:** dense `numpy` float32 arrays with `nan` where OSRM returns no route, plus a small JSON sidecar listing pharmacy `place_id`s in column order. A Folium **heatmap-style raster** (`osrm_best_drive_time_map.html`) shows best drive time across the **five-county** study region (`counties_five_vt.geojson`, including Essex) with pharmacy markers.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from rich.progress import BarColumn, Progress, TaskProgressColumn, TextColumn, TimeElapsedColumn, TimeRemainingColumn

DATA_DIR = Path("../data")
ADDRESSES_CSV = DATA_DIR / "addresses_with_population_weights.csv"
PHARMACIES_JSON = DATA_DIR / "pharmacies_active.json"

OSRM_BASE = "http://127.0.0.1:8008"
MAX_TABLE_COORDS = 100

OUT_DURATION = DATA_DIR / "osrm_address_pharmacy_duration_sec.npy"
OUT_DISTANCE = DATA_DIR / "osrm_address_pharmacy_distance_m.npy"
OUT_META = DATA_DIR / "osrm_address_pharmacy_matrix_meta.json"

PHARM_BATCH = 50
REQUEST_TIMEOUT_S = 300

In [ ]:
addresses = pd.read_csv(ADDRESSES_CSV)
with PHARMACIES_JSON.open() as f:
    pharmacies = json.load(f)

n_addr = len(addresses)
n_pharm = len(pharmacies)
print(f"{n_addr} addresses, {n_pharm} pharmacies")

addr_lon = addresses["lon"].to_numpy(dtype=np.float64)
addr_lat = addresses["lat"].to_numpy(dtype=np.float64)
pharm_lon = np.array([p["lon"] for p in pharmacies], dtype=np.float64)
pharm_lat = np.array([p["lat"] for p in pharmacies], dtype=np.float64)
pharm_place_ids = [p["place_id"] for p in pharmacies]

In [ ]:
def osrm_table_block(
    alon: np.ndarray,
    alat: np.ndarray,
    plon: np.ndarray,
    plat: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Return duration (sec) and distance (m) matrices, shape (n_addr, n_pharm)."""
    n_a, n_p = len(alon), len(plon)
    if n_a + n_p > MAX_TABLE_COORDS:
        raise ValueError(f"block too large: {n_a} + {n_p} > {MAX_TABLE_COORDS}")
    coords = ";".join(
        [f"{lon},{lat}" for lon, lat in zip(alon, alat)]
        + [f"{lon},{lat}" for lon, lat in zip(plon, plat)]
    )
    sources = ";".join(str(i) for i in range(n_a))
    destinations = ";".join(str(i) for i in range(n_a, n_a + n_p))
    url = f"{OSRM_BASE}/table/v1/driving/{coords}"
    r = requests.get(
        url,
        params={
            "sources": sources,
            "destinations": destinations,
            "annotations": "duration,distance",
        },
        timeout=REQUEST_TIMEOUT_S,
    )
    r.raise_for_status()
    data = r.json()
    if data.get("code") != "Ok":
        raise RuntimeError(data)

    def to_float32_matrix(rows: list) -> np.ndarray:
        out = np.empty((len(rows), len(rows[0])), dtype=np.float32)
        for i, row in enumerate(rows):
            for j, v in enumerate(row):
                out[i, j] = np.nan if v is None else float(v)
        return out

    return to_float32_matrix(data["durations"]), to_float32_matrix(data["distances"])

In [ ]:
duration = np.full((n_addr, n_pharm), np.nan, dtype=np.float32)
distance = np.full((n_addr, n_pharm), np.nan, dtype=np.float32)

req_n = 0
for p_start in range(0, n_pharm, PHARM_BATCH):
    p_end = min(p_start + PHARM_BATCH, n_pharm)
    pc = p_end - p_start
    addr_batch = MAX_TABLE_COORDS - pc
    for a_start in range(0, n_addr, addr_batch):
        req_n += 1

print(f"{req_n:,} OSRM table requests")
columns = (
    TextColumn("[bold]OSRM[/]"),
    BarColumn(bar_width=None),
    TaskProgressColumn(),
    TimeElapsedColumn(),
    TimeRemainingColumn(),
)
with Progress(*columns, transient=False) as progress:
    task = progress.add_task("table requests", total=req_n)
    for p_start in range(0, n_pharm, PHARM_BATCH):
        p_end = min(p_start + PHARM_BATCH, n_pharm)
        pc = p_end - p_start
        addr_batch = MAX_TABLE_COORDS - pc
        plon = pharm_lon[p_start:p_end]
        plat = pharm_lat[p_start:p_end]
        for a_start in range(0, n_addr, addr_batch):
            a_end = min(a_start + addr_batch, n_addr)
            alon = addr_lon[a_start:a_end]
            alat = addr_lat[a_start:a_end]
            d_sec, d_m = osrm_table_block(alon, alat, plon, plat)
            duration[a_start:a_end, p_start:p_end] = d_sec
            distance[a_start:a_end, p_start:p_end] = d_m
            progress.advance(task)

np.save(OUT_DURATION, duration)
np.save(OUT_DISTANCE, distance)
meta = {
    "osrm_base": OSRM_BASE,
    "n_addresses": n_addr,
    "n_pharmacies": n_pharm,
    "pharmacy_place_ids": pharm_place_ids,
    "address_csv": str(ADDRESSES_CSV.resolve()),
    "pharmacies_json": str(PHARMACIES_JSON.resolve()),
    "row_major": "duration[i, j] is address row i to pharmacy column j",
}
OUT_META.write_text(json.dumps(meta, indent=2))
print(f"Wrote {OUT_DURATION.name}, {OUT_DISTANCE.name}, {OUT_META.name}")
print(
    "finite duration cells:",
    np.isfinite(duration).sum(),
    "/",
    duration.size,
)

## Quick check

Reload from disk after a long run (defines paths locally so this cell stands alone).

In [ ]:
from pathlib import Path

import json
import numpy as np

DATA_DIR = Path("../data")
OUT_DURATION = DATA_DIR / "osrm_address_pharmacy_duration_sec.npy"
OUT_META = DATA_DIR / "osrm_address_pharmacy_matrix_meta.json"

d = np.load(OUT_DURATION)
meta = json.loads(OUT_META.read_text())
print(d.shape, meta["n_addresses"], meta["n_pharmacies"])
mins = np.nanmin(d, axis=1)
print("addresses with at least one route:", np.isfinite(mins).sum())
print("median best-time (min) among reachable:", np.nanmedian(mins) / 60)

## Best drive time map (OSRM)

Interpolated surface (same approach as notebook `01-pharmacy_access`): **scipy `griddata`** (linear, with nearest fill for gaps) from every address point’s **best** pharmacy time (`nanmin` over the OSRM matrix). **Pharmacies** are drawn as markers. Individual address dots are omitted—there are ~87k points; the raster carries the detail.

Mask and county outlines use **`../data/counties_five_vt.geojson`** (TIGER/Line 2023, 500k): Caledonia, **Essex**, Lamoille, Orleans, and Washington—matching the five-county address sample in notebook `02-people_served`. Falls back to `counties.geojson` or the older `target_region.geojson` if needed.

Writes `../data/osrm_best_drive_time_map.html` so the map opens without re-running the matrix step.

In [ ]:
import json
from pathlib import Path

import branca.colormap as cm
import folium
import geopandas as gpd
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import shapely
from scipy.interpolate import griddata
from shapely.ops import unary_union

DATA_DIR = Path("../data")
ADDRESSES_CSV = DATA_DIR / "addresses_with_population_weights.csv"
PHARMACIES_JSON = DATA_DIR / "pharmacies_active.json"
OUT_DURATION = DATA_DIR / "osrm_address_pharmacy_duration_sec.npy"
FIVE_COUNTIES_GEOJSON = DATA_DIR / "counties_five_vt.geojson"
LEGACY_COUNTIES_GEOJSON = DATA_DIR / "counties.geojson"
LEGACY_REGION_GEOJSON = DATA_DIR / "target_region.geojson"
MAP_HTML = DATA_DIR / "osrm_best_drive_time_map.html"

d_matrix = np.load(OUT_DURATION)
addresses_df = pd.read_csv(ADDRESSES_CSV)
if len(addresses_df) != d_matrix.shape[0]:
    raise ValueError("Address row count does not match duration matrix rows")

min_sec = np.nanmin(d_matrix, axis=1)
drive_times_df = addresses_df.assign(min_drive_minutes=min_sec / 60.0)

with PHARMACIES_JSON.open() as f:
    pharm_records = json.load(f)
pharmacies_df = pd.DataFrame(pharm_records)

if FIVE_COUNTIES_GEOJSON.exists():
    boundary_gdf = gpd.read_file(FIVE_COUNTIES_GEOJSON)
    boundary_name = "County boundaries"
elif LEGACY_COUNTIES_GEOJSON.exists():
    boundary_gdf = gpd.read_file(LEGACY_COUNTIES_GEOJSON)
    boundary_name = "County boundaries"
else:
    boundary_gdf = gpd.read_file(LEGACY_REGION_GEOJSON)
    boundary_name = "Study region"

region = unary_union(boundary_gdf.geometry)
minx, miny, maxx, maxy = region.bounds

GRID_RES = 300
lon_grid = np.linspace(minx, maxx, GRID_RES)
lat_grid = np.linspace(miny, maxy, GRID_RES)
lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)

pts = drive_times_df[["lon", "lat"]].to_numpy()
vals = drive_times_df["min_drive_minutes"].to_numpy()

grid_z = griddata(pts, vals, (lon_mesh, lat_mesh), method="linear")
grid_nn = griddata(pts, vals, (lon_mesh, lat_mesh), method="nearest")
grid_z = np.where(np.isnan(grid_z), grid_nn, grid_z)
grid_z = np.clip(grid_z, 0, None)

grid_pts = shapely.points(lon_mesh.ravel(), lat_mesh.ravel())
inside = shapely.within(grid_pts, region).reshape(lon_mesh.shape)
grid_z[~inside] = np.nan

CMAP_COLORS = ["#2ecc71", "#82e0aa", "#f9e79f", "#f0b27a", "#e74c3c", "#8b0000"]
CMAP_BREAKS = np.array([0, 10, 15, 20, 30, 45, 60], dtype=float)
CMAP_TARGETS = np.linspace(0, 1, len(CMAP_BREAKS))

_crgba = [mcolors.to_rgba(c) for c in CMAP_COLORS]
_pos = np.linspace(0, 1, len(CMAP_COLORS))
_cdict = {
    ch: [(_pos[i], _crgba[i][j], _crgba[i][j]) for i in range(len(_crgba))]
    for j, ch in enumerate(["red", "green", "blue"])
}
drive_cmap = mcolors.LinearSegmentedColormap("drive_time", _cdict)

normed = np.interp(np.clip(grid_z, 0, 60), CMAP_BREAKS, CMAP_TARGETS)
rgba = drive_cmap(normed)
rgba[..., 3] = np.where(np.isnan(grid_z), 0.0, 0.45)
rgba = np.flipud(rgba)

center_lat = drive_times_df["lat"].mean()
center_lon = drive_times_df["lon"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=9, tiles="cartodbpositron")

boundary_kwargs: dict = {
    "name": boundary_name,
    "style_function": lambda _: {"fillOpacity": 0, "color": "#333333", "weight": 2},
}
if "county_name" in boundary_gdf.columns:
    boundary_kwargs["tooltip"] = folium.GeoJsonTooltip(
        fields=["county_name"], aliases=["County"]
    )
folium.GeoJson(boundary_gdf.__geo_interface__, **boundary_kwargs).add_to(m)

folium.raster_layers.ImageOverlay(
    image=rgba,
    opacity=0.6,
    bounds=[[miny, minx], [maxy, maxx]],
    name="Best drive time (OSRM)",
    interactive=False,
    zindex=1,
).add_to(m)

pharm_layer = folium.FeatureGroup(name="Pharmacies")
for _, row in pharmacies_df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6,
        color="#2c3e50",
        fill=True,
        fill_color="#3498db",
        fill_opacity=0.9,
        weight=1,
        tooltip=f"{row['name']} ({row['city']})",
    ).add_to(pharm_layer)
pharm_layer.add_to(m)

colormap = cm.LinearColormap(
    colors=CMAP_COLORS + ["#8b0000"],
    index=[0, 10, 15, 20, 30, 45, 60],
    vmin=0,
    vmax=60,
    caption="Best drive time to any pharmacy (minutes, OSRM)",
)
colormap.add_to(m)

folium.LayerControl().add_to(m)
m.save(str(MAP_HTML))
print(f"Saved {MAP_HTML}")
m